CPU-only rescue. Run AFTER the original full session ends and outputs are saved. HF_TOKEN access is required. This notebook imports matching screening files, commits them to Hugging Face and downloads them again to verify integrity. It performs no audio screening or model training. If outputs are missing, it stops.

In [ ]:
import os,sys,json,logging,subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
subprocess.run([sys.executable,'-m','pip','install','--quiet','huggingface_hub==0.36.2'],check=True)
ROOT=Path('/kaggle/working/recovery-tools');ROOT.mkdir(exist_ok=True)
(ROOT/'orpheus_utils.py').write_text('"""CPU-only data/token/artifact utilities; no CUDA imports at module scope."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport logging\nimport os\nimport re\nimport subprocess\nimport tarfile\nimport tempfile\nimport time\nimport unicodedata\nimport zipfile\nfrom pathlib import Path\n\nSPECIAL = dict(end_text=128009, start_speech=128257, end_speech=128258,\n               start_human=128259, end_human=128260, start_ai=128261,\n               end_ai=128262, pad=128263)\nAUDIO_BASE = 128266\nOFFSETS = [4096 * i for i in range(7)]\nCODEC_ID = "hubertsiuzdak/snac_24khz"\nFORMAT_VERSION = 1\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\ndef fingerprint(value):\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    with tmp.open("w", encoding="utf-8") as f:\n        f.write(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n")\n        f.flush()\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef append_jsonl(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as f:\n        f.write(json.dumps(value, ensure_ascii=False, allow_nan=False) + "\\n")\n        f.flush()\n\n\ndef clean_text(text):\n    text = unicodedata.normalize("NFC", str(text))\n    text = "".join(c for c in text if unicodedata.category(c) != "Cc" or c in "\\t\\n")\n    return " ".join(text.split())\n\n\ndef contained(root, relative):\n    root = Path(root).resolve()\n    # Reject backslashes too: archives may be made on Windows.\n    p = Path(str(relative).replace("\\\\", "/"))\n    if p.is_absolute() or ".." in p.parts or re.match(r"^[A-Za-z]:", str(p)):\n        raise ValueError(f"Unsafe relative path: {relative}")\n    resolved = (root / p).resolve()\n    if not resolved.is_relative_to(root):\n        raise ValueError(f"Path escapes dataset: {relative}")\n    return resolved\n\n\ndef safe_unzip(archive, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for member in z.infolist():\n            contained(destination, member.filename)\n            if (member.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are unsupported")\n        z.extractall(destination)\n\n\ndef safe_untar(archive, destination):\n    destination = Path(destination)\n    with tarfile.open(archive, "r:gz") as t:\n        for m in t.getmembers():\n            contained(destination, m.name)\n            if not (m.isfile() or m.isdir()):\n                raise ValueError(f"Unsupported TAR member: {m.name}")\n        # Members validated above; filter also protects modern Python runtimes.\n        t.extractall(destination, filter="data")\n\n\ndef prompt_ids(tokenizer, text, speaker=None):\n    text = clean_text(text)\n    if not text:\n        raise ValueError("Empty synthesis text")\n    prompt = f"{speaker}: {text}" if speaker else text\n    return ([SPECIAL["start_human"]] + tokenizer.encode(prompt, add_special_tokens=True)\n            + [SPECIAL["end_text"], SPECIAL["end_human"], SPECIAL["start_ai"],\n               SPECIAL["start_speech"]])\n\n\ndef interleave_codes(c0, c1, c2, dedup="none"):\n    if not c0 or len(c1) != 2 * len(c0) or len(c2) != 4 * len(c0):\n        raise ValueError("Unexpected SNAC codebook lengths")\n    if any(not 0 <= v < 4096 for codes in (c0, c1, c2) for v in codes):\n        raise ValueError("SNAC code outside codebook")\n    if dedup not in ("none", "coarse"):\n        raise ValueError("Unknown frame deduplication policy")\n    result, previous = [], None\n    for i, v in enumerate(c0):\n        frame = [v, c1[2*i], c2[4*i], c2[4*i+1], c1[2*i+1], c2[4*i+2], c2[4*i+3]]\n        if dedup == "none" or previous is None or v != previous:\n            result.extend(AUDIO_BASE + v + offset for v, offset in zip(frame, OFFSETS))\n            previous = frame[0]\n    return result\n\n\ndef training_row(prefix, audio_ids, objective, max_length, vocab_size):\n    ids = prefix + audio_ids + [SPECIAL["end_speech"], SPECIAL["end_ai"]]\n    if len(ids) > max_length:\n        raise ValueError(f"Sequence has {len(ids)} tokens > {max_length}; increase max_length, rebuild cache")\n    if min(ids) < 0 or max(ids) >= vocab_size:\n        raise ValueError("Token ID outside model vocabulary")\n    if objective not in ("all", "audio"):\n        raise ValueError("Unknown training objective")\n    labels = ids.copy()\n    if objective == "audio":\n        # Learn start_speech, all audio, and both stop delimiters.\n        labels[:len(prefix)-1] = [-100] * (len(prefix)-1)\n    return dict(input_ids=ids, labels=labels, attention_mask=[1]*len(ids), length=len(ids))\n\n\ndef decode_frames(ids):\n    stopped = SPECIAL["end_speech"] in ids\n    if stopped:\n        ids = ids[:ids.index(SPECIAL["end_speech"])]\n    frames, invalid = [], None\n    for start in range(0, len(ids)-6, 7):\n        f = [v - AUDIO_BASE - offset for v, offset in zip(ids[start:start+7], OFFSETS)]\n        if any(not 0 <= v < 4096 for v in f):\n            invalid = start\n            break\n        frames.append(f)\n    status = dict(ended_with_speech_stop=stopped, generated_tokens=len(ids),\n                  complete_frames=len(frames), invalid_frame_token_offset=invalid,\n                  trailing_tokens=len(ids) % 7)\n    if not frames:\n        raise ValueError(f"No valid SNAC frames; first tokens: {ids[:14]}")\n    codes = [[f[0] for f in frames], [v for f in frames for v in (f[1], f[4])],\n             [v for f in frames for v in (f[2], f[3], f[5], f[6])]]\n    return codes, status\n\n\ndef seal_checkpoint(checkpoint):\n    checkpoint = Path(checkpoint)\n    required = ["adapter_config.json", "adapter_model.safetensors", "trainer_state.json",\n                "optimizer.pt", "scheduler.pt", "training_args.bin", "rng_state.pth"]\n    missing = [n for n in required if not (checkpoint / n).is_file()]\n    if missing:\n        raise RuntimeError(f"Checkpoint is not resumable: missing {missing}")\n    files = {str(p.relative_to(checkpoint)): sha256(p) for p in sorted(checkpoint.rglob("*"))\n             if p.is_file() and p.name != "COMPLETE.json" and not p.name.endswith(".tmp")}\n    atomic_json(checkpoint / "COMPLETE.json", dict(files=files))\n\n\ndef verify_checkpoint(checkpoint):\n    checkpoint = Path(checkpoint)\n    marker = checkpoint / "COMPLETE.json"\n    if not marker.is_file():\n        return False\n    files = read_json(marker)["files"]\n    required = {"adapter_config.json", "adapter_model.safetensors", "trainer_state.json",\n                "optimizer.pt", "scheduler.pt", "training_args.bin", "rng_state.pth"}\n    if not isinstance(files, dict) or not required <= set(files):\n        raise RuntimeError(f"Incomplete checkpoint manifest: {checkpoint}")\n    for relative, digest in files.items():\n        p = contained(checkpoint, relative)\n        if not p.is_file() or sha256(p) != digest:\n            raise RuntimeError(f"Checkpoint checksum failed: {p}")\n    return True\n\n\ndef latest_checkpoint(run):\n    candidates = sorted((p for p in Path(run).glob("checkpoint-*")\n                         if p.is_dir() and p.name.split("-")[-1].isdigit()),\n                        key=lambda p: int(p.name.split("-")[-1]), reverse=True)\n    damaged = []\n    for p in candidates:\n        try:\n            if verify_checkpoint(p):\n                if damaged:\n                    logging.getLogger("orpheus_urdu").warning(\n                        "Ignoring damaged checkpoints %s; resuming verified %s", damaged, p.name)\n                return p\n        except (OSError, ValueError, KeyError, TypeError, RuntimeError):\n            damaged.append(p.name)\n    if damaged:\n        raise RuntimeError(f"No verified checkpoint remains; damaged candidates: {damaged}. "\n                           "Restore a remote copy before resuming")\n    return None\n\n\nclass SnapshotStore:\n    """Verified artifacts shared by local, Drive and Hugging Face storage."""\n    def __init__(self, remote=None):\n        self.remote = remote\n\n    def preflight(self):\n        if self.remote:\n            # Write AND read a probe before allocating GPU time.\n            with tempfile.TemporaryDirectory() as d:\n                probe = Path(d) / "probe.json"\n                atomic_json(probe, {"time": time.time()})\n                self.put(probe, "connection_probe.json")\n                out = Path(d) / "readback.json"\n                self.get("connection_probe.json", out)\n                if sha256(probe) != sha256(out):\n                    raise RuntimeError("Remote storage probe checksum mismatch")\n\n    def put(self, local, relative):\n        if self.remote:\n            raise NotImplementedError\n\n    def put_many(self, items, lane="files"):\n        """Upload [(local, relative), ...]; Hub overrides this with one commit."""\n        for local, relative in items:\n            self.put(local, relative)\n\n    def get(self, relative, local):\n        raise NotImplementedError\n\n    def names(self):\n        return []\n\n    def restore_files(self, prefix, destination, skip_existing=False):\n        """Restore a preparation folder or immutable cache without backend-specific calls."""\n        if not self.remote:\n            return\n        prefix = prefix.strip("/") + "/"\n        destination = Path(destination)\n        destination.mkdir(parents=True, exist_ok=True)\n        for name in sorted(self.names()):\n            if not name.startswith(prefix) or ".tmp" in Path(name).name:\n                continue\n            local = contained(destination, name[len(prefix):])\n            if skip_existing and local.exists():\n                continue\n            local.parent.mkdir(parents=True, exist_ok=True)\n            temp = local.with_name(local.name + ".download.tmp")\n            try:\n                self.get(name, temp)\n                temp.replace(local)\n            finally:\n                temp.unlink(missing_ok=True)\n\n    def restore(self, run):\n        if not self.remote:\n            return\n        if "latest.json" not in self.names():\n            # Encoding may have been interrupted BEFORE any Trainer checkpoint existed.\n            self.restore_files("preparation", run, skip_existing=True)\n            return\n        with tempfile.TemporaryDirectory(dir=Path(run).parent) as d:\n            pointer = Path(d) / "latest.json"\n            self.get("latest.json", pointer)\n            info = read_json(pointer)\n            archive = Path(d) / "snapshot.tar.gz"\n            self.get(info["archive"], archive)\n            if sha256(archive) != info["sha256"]:\n                raise RuntimeError("Remote snapshot checksum mismatch")\n            staging = Path(d) / "staging"\n            staging.mkdir()\n            safe_untar(archive, staging)\n            cp = staging / info["checkpoint"]\n            if not verify_checkpoint(cp):\n                raise RuntimeError("Remote checkpoint missing completion marker")\n            # Refuse to overwrite newer local progress.\n            local = latest_checkpoint(run)\n            if local and int(local.name.split("-")[-1]) >= info["step"]:\n                return\n            import shutil\n            shutil.copytree(staging, run, dirs_exist_ok=True)\n\n    def backup(self, run, checkpoint):\n        if not self.remote:\n            return\n        run, checkpoint = Path(run), Path(checkpoint)\n        if not verify_checkpoint(checkpoint):\n            raise RuntimeError("Refusing to upload incomplete checkpoint")\n        with tempfile.TemporaryDirectory(dir=run.parent) as d:\n            archive = Path(d) / "snapshot.tar.gz"\n            with tarfile.open(archive, "w:gz", compresslevel=1, dereference=True) as t:\n                for p in sorted(run.iterdir()):\n                    if p.name.startswith("checkpoint-") and p != checkpoint:\n                        continue\n                    if p.name in (".lock",) or p.name.endswith(".tmp"):\n                        continue\n                    t.add(p, arcname=p.name)\n            digest = sha256(archive)\n            remote_name = f"snapshots/{checkpoint.name}-{digest[:12]}.tar.gz"\n            pointer = Path(d) / "latest.json"\n            atomic_json(pointer, dict(archive=remote_name, sha256=digest,\n                                      checkpoint=checkpoint.name,\n                                      step=int(checkpoint.name.split("-")[-1])))\n            self.publish_snapshot(archive, remote_name, pointer)\n\n    def publish_snapshot(self, archive, remote_name, pointer):\n        # Drive uploads the pointer only AFTER the immutable archive succeeds.\n        self.put(archive, remote_name)\n        self.put(pointer, "latest.json")\n\n\nclass DriveStore(SnapshotStore):\n    """rclone OAuth credentials live outside experiment artifacts. No remote deletes."""\n    def __init__(self, remote=None, config=None):\n        super().__init__(remote.rstrip("/") if remote else None)\n        self.config = config\n\n    def run(self, *args):\n        cmd = ["rclone"]\n        if self.config:\n            cmd += ["--config", str(self.config)]\n        cmd += ["--retries", "3", "--low-level-retries", "10", "--contimeout", "30s",\n                "--timeout", "5m", "--log-level", "ERROR", *map(str, args)]\n        # Do not print stderr: a backend error could include sensitive token/config fields.\n        r = subprocess.run(cmd, capture_output=True, text=True)\n        if r.returncode:\n            raise RuntimeError(f"Drive operation {args[0]} failed (code {r.returncode}); check access/quota/network")\n        return r.stdout\n\n    def put(self, local, relative):\n        if self.remote:\n            self.run("copyto", local, self.remote + "/" + relative)\n\n    def get(self, relative, local):\n        Path(local).parent.mkdir(parents=True, exist_ok=True)\n        self.run("copyto", self.remote + "/" + relative, local)\n\n    def names(self):\n        if not self.remote:\n            return []\n        return self.run("lsf", self.remote, "--files-only", "--recursive").splitlines()\n\n    def restore_files(self, prefix, destination, skip_existing=False):\n        if self.remote and any(n.startswith(prefix.rstrip("/") + "/") for n in self.names()):\n            options = ["--ignore-existing"] if skip_existing else []\n            self.run("copy", self.remote + "/" + prefix.strip("/"), destination,\n                     "--exclude", "*.tmp*", *options)\n\n\n_SECRET_PATTERNS = (\n    re.compile(r"hf_[A-Za-z0-9]{6,}"),\n    re.compile(r"(?i)\\b(bearer|basic)\\s+[A-Za-z0-9._~+/=-]+"),\n    re.compile(r"(?i)\\b(token|access_token|password|authorization)([=:]\\s*)[^\\s&\\"\',]+"),\n    re.compile(r"://[^/\\s:@]+:[^/\\s@]+@"),\n)\n\n\ndef redact(text, *secrets, limit=800):\n    """Remove Hub tokens and auth headers before text reaches logs or reports."""\n    text = str(text)\n    for secret in secrets:\n        if secret and len(secret) >= 6:\n            text = text.replace(secret, "***")\n    text = _SECRET_PATTERNS[0].sub("hf_***", text)\n    text = _SECRET_PATTERNS[1].sub(lambda m: m.group(1) + " ***", text)\n    text = _SECRET_PATTERNS[2].sub(lambda m: m.group(1) + m.group(2) + "***", text)\n    text = _SECRET_PATTERNS[3].sub("://***@", text)\n    return text if len(text) <= limit else text[:limit] + "...[truncated]"\n\n\nHUB_HINTS = {\n    "authentication": "HF_TOKEN was rejected (invalid, expired, revoked or not sent). Create a new token and update the Kaggle Secret.",\n    "permission": "HF_TOKEN is valid but may not write to this repository. Use a \'write\' token, or a fine-grained token with \'Write access to contents/settings of all repos\' or this repo selected.",\n    "not_found": "Repository not visible to this token: check hf_repo_id spelling/owner and that a fine-grained token includes this repository.",\n    "quota": "Hub storage or usage quota appears exceeded; free space in the account or use a different repository.",\n    "rate_limit": "Hub rate limit reached; retry later or reduce commit frequency.",\n    "payload_too_large": "File exceeds a Hub request size limit.",\n    "conflict": "Concurrent Hub commit conflict; normally transient.",\n    "server": "Hugging Face server error; normally transient (see status.huggingface.co).",\n    "connectivity": "Network failure reaching huggingface.co: check Kaggle Internet setting, proxy/DNS and HF_ENDPOINT.",\n    "offline_mode": "HF_HUB_OFFLINE is set; unset it to allow uploads.",\n    "api_request": "Hub rejected the request; see server_message.",\n    "client_or_local": "Error raised locally by the Hub client before/without an HTTP response; see exception_type and message.",\n    "timeout": "Upload exceeded its time limit (stalled connection or very slow transfer); it stays queued locally and is retried later.",\n}\n_RETRYABLE = {"rate_limit", "conflict", "server", "connectivity", "client_or_local"}\n\n\nclass HubOperationError(RuntimeError):\n    """Redacted Hub failure; .diagnostics holds JSON-safe evidence for reports."""\n    def __init__(self, message, diagnostics):\n        super().__init__(message)\n        self.diagnostics = diagnostics\n\n\ndef _header(response, name):\n    try:\n        return (getattr(response, "headers", None) or {}).get(name)\n    except Exception:\n        return None\n\n\ndef describe_hub_error(exc, operation, *secrets):\n    response = getattr(exc, "response", None)\n    status = getattr(response, "status_code", None)\n    status = status if isinstance(status, int) else None\n    server_message = getattr(exc, "server_message", None) or _header(response, "x-error-message")\n    if not server_message and response is not None:\n        try:\n            body = response.json()\n            server_message = body.get("error") if isinstance(body, dict) else None\n        except Exception:\n            server_message = None\n    chain, seen = [], exc\n    while seen is not None and len(chain) < 6:\n        chain.append(seen)\n        seen = seen.__cause__ or seen.__context__\n    names = [c.__name__ for e in chain for c in type(e).__mro__]\n    message = redact(exc, *secrets)\n    text = f"{server_message or \'\'} {message}".lower()\n    if status is None:\n        if "OfflineModeIsEnabled" in names:\n            category = "offline_mode"\n        elif any(k in n for n in names for k in ("ConnectionError", "ConnectError", "Timeout", "SSLError",\n                                                   "ProxyError", "NameResolution", "socket")):\n            category = "connectivity"\n        else:\n            category = "client_or_local"\n    elif status == 507 or any(k in text for k in ("quota", "storage limit", "insufficient storage",\n                                                    "exceeded your", "storage space")):\n        category = "quota"\n    else:\n        category = {401: "authentication", 403: "permission", 404: "not_found", 408: "connectivity",\n                    409: "conflict", 412: "conflict", 413: "payload_too_large",\n                    429: "rate_limit"}.get(status, "server" if status >= 500 else "api_request")\n    retry_after = _header(response, "retry-after")\n    rate_limit = _header(response, "ratelimit")\n    if retry_after is None and rate_limit:\n        # Hub format (IETF draft): "api";r=0;t=<seconds until reset>\n        match = re.search(r"\\bt=(\\d+)", str(rate_limit))\n        retry_after = match.group(1) if match else None\n    return dict(operation=operation, category=category, http_status=status,\n                exception_type=f"{type(exc).__module__}.{type(exc).__qualname__}",\n                exception_chain=[f"{type(e).__module__}.{type(e).__qualname__}" for e in chain[1:]],\n                server_message=redact(server_message, *secrets) if server_message else None,\n                request_id=_header(response, "x-request-id"), error_code=_header(response, "x-error-code"),\n                retry_after=retry_after, rate_limit=rate_limit, message=message, hint=HUB_HINTS[category])\n\n\ndef hub_token_report(whoami, repo_id):\n    """Summarise token scope from HfApi.whoami(); can_write None means undetermined."""\n    namespace = repo_id.split("/")[0]\n    access = ((whoami or {}).get("auth") or {}).get("accessToken") or {}\n    role = access.get("role")\n    orgs = [o.get("name") for o in (whoami or {}).get("orgs") or [] if isinstance(o, dict)]\n    report = dict(user=(whoami or {}).get("name"), token_role=role, token_name=access.get("displayName"),\n                  namespace_is_user_or_org=namespace in [(whoami or {}).get("name"), *orgs], can_write=None)\n    if role == "write":\n        report["can_write"] = True\n    elif role == "read":\n        report["can_write"] = False\n    elif role == "fineGrained":\n        scoped = (access.get("fineGrained") or {}).get("scoped") or []\n        grants = []\n        for entry in scoped:\n            entity = entry.get("entity") or {}\n            if entity.get("name") in (namespace, repo_id):\n                grants += entry.get("permissions") or []\n        report["fine_grained_permissions_for_repo"] = sorted(set(grants))\n        report["can_write"] = "repo.write" in grants\n    return report\n\n\nclass HuggingFaceStore(SnapshotStore):\n    """Private Hub model repo; synchronous commits and bounded transfer retries.\n\n    Training snapshots and caches are stored under runs/<run_id>/. This is an\n    artifact repository, not a directly loadable push_to_hub model directory.\n    Dependencies are imported lazily so CPU tests/local/Drive do not need Hub auth.\n    """\n    def __init__(self, repo_id, run_id, token=None, *, api=None, download=None, commit_add=None):\n        if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repo_id or ""):\n            raise ValueError("hf_repo_id must be USERNAME/REPOSITORY")\n        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,100}", run_id):\n            raise ValueError("Invalid run_id")\n        if api is None or download is None or commit_add is None:\n            from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd\n            api = api if api is not None else HfApi(token=token)\n            download = download if download is not None else hf_hub_download\n            commit_add = commit_add if commit_add is not None else CommitOperationAdd\n        self.repo_id, self.prefix = repo_id, f"runs/{run_id}"\n        super().__init__(f"hf://{repo_id}/{self.prefix}")\n        self.api, self._download, self._commit_add = api, download, commit_add\n        self._token, self._files = token, None\n        self.token_report, self.last_error = None, None\n        self._inflight = {}\n\n    # Hub/Xet transfers have no overall deadline; a stalled one must not freeze training.\n    UPLOAD_TIMEOUTS = {"checkpoint": 1800, "files": 600, "heartbeat": 300}\n\n    def _bounded(self, lane, operation, function):\n        import threading\n        previous = self._inflight.get(lane)\n        if previous is not None and previous.is_alive():\n            raise HubOperationError(f"Hugging Face {operation} skipped: the previous {lane} upload is still "\n                                    "running in the background", self._timeout_info(operation, still_running=True))\n        result = {}\n\n        def work():\n            try:\n                result["value"] = function()\n            except BaseException as exc:\n                result["error"] = exc\n        thread = threading.Thread(target=work, name=f"hub-{lane}", daemon=True)\n        self._inflight[lane] = thread\n        thread.start()\n        thread.join(self.UPLOAD_TIMEOUTS[lane])\n        if thread.is_alive():\n            info = self._timeout_info(operation)\n            self.last_error = info\n            logging.getLogger("orpheus_urdu").warning("Hugging Face %s timed out: %s", operation, json.dumps(info))\n            raise HubOperationError(f"Hugging Face {operation} timed out after {self.UPLOAD_TIMEOUTS[lane]}s. "\n                                    f"{HUB_HINTS[\'timeout\']}", info)\n        if "error" in result:\n            raise result["error"]\n        return result.get("value")\n\n    def _timeout_info(self, operation, still_running=False):\n        return dict(operation=operation, category="timeout", http_status=None, exception_type="TimeoutError",\n                    exception_chain=[], server_message=None, request_id=None, error_code=None,\n                    retry_after=None, rate_limit=None, still_running=still_running,\n                    message="previous upload still running" if still_running else "upload deadline exceeded",\n                    hint=HUB_HINTS["timeout"])\n\n    def _call(self, operation, function, *, missing_ok=False, attempts=3, **kwargs):\n        log = logging.getLogger("orpheus_urdu")\n        for attempt in range(1, attempts + 1):\n            try:\n                return function(**kwargs)\n            except Exception as exc:\n                info = describe_hub_error(exc, operation, self._token)\n                if missing_ok and info["http_status"] == 404:\n                    return None\n                info["attempt"] = f"{attempt}/{attempts}"\n                if self.token_report and info["category"] in ("authentication", "permission", "not_found"):\n                    info["token_report"] = self.token_report\n                self.last_error = info\n                log.warning("Hugging Face %s attempt %d/%d failed: %s", operation, attempt, attempts,\n                            json.dumps(info, ensure_ascii=False))\n                if info["category"] not in _RETRYABLE or attempt == attempts:\n                    # Original traceback suppressed: request objects may carry auth headers.\n                    # The redacted evidence above and in .diagnostics replaces it.\n                    raise HubOperationError(\n                        f"Hugging Face {operation} failed [{info[\'category\']}] HTTP {info[\'http_status\']} "\n                        f"{info[\'exception_type\']}: {info[\'server_message\'] or info[\'message\']} "\n                        f"(request id {info[\'request_id\']}). {info[\'hint\']}"\n                        + (f" Token scope: {json.dumps(self.token_report)}" if "token_report" in info else ""),\n                        info) from None\n                delay = 2 ** attempt\n                if info["category"] == "rate_limit":\n                    # Waiting out a long limit here would stall encoding/training and further\n                    # retries keep the limit tripped; callers\' deferred queue backs off instead.\n                    try:\n                        delay = max(float(info["retry_after"]), 5)\n                    except (TypeError, ValueError):\n                        delay = None\n                    if delay is None or delay > 60:\n                        raise HubOperationError(\n                            f"Hugging Face {operation} rate limited (HTTP 429, retry-after "\n                            f"{info[\'retry_after\']}): {info[\'server_message\'] or info[\'message\']}. {info[\'hint\']}",\n                            info) from None\n                time.sleep(delay)\n\n    def inspect_token(self):\n        """Record token role/scope as evidence; never fatal except a rejected token."""\n        whoami = getattr(self.api, "whoami", None)\n        if whoami is None:\n            return None\n        try:\n            # One attempt: whoami is rate limited and this check is diagnostic only.\n            self.token_report = hub_token_report(self._call("token inspection", whoami, attempts=1), self.repo_id)\n        except HubOperationError as exc:\n            if exc.diagnostics["category"] == "authentication":\n                raise\n            return None\n        log = logging.getLogger("orpheus_urdu")\n        log.info("HF token: %s", json.dumps(self.token_report))\n        if self.token_report["can_write"] is False:\n            log.warning("HF_TOKEN appears unable to write to %s; the probe upload will confirm", self.repo_id)\n        return self.token_report\n\n    def _path(self, relative):\n        # Validate the remote artifact path as well as local extraction paths.\n        contained(Path("/tmp/orpheus-hub-paths"), relative)\n        return self.prefix + "/" + relative\n\n    def preflight(self):\n        log = logging.getLogger("orpheus_urdu")\n        try:\n            import importlib.metadata as md\n            versions = {}\n            for name in ("huggingface_hub", "hf_xet", "hf_transfer", "requests", "httpx"):\n                try:\n                    versions[name] = md.version(name)\n                except md.PackageNotFoundError:\n                    versions[name] = None\n            env = {k: redact(os.environ[k]) for k in ("HF_ENDPOINT", "HF_HUB_OFFLINE", "HF_HUB_ENABLE_HF_TRANSFER",\n                                                     "HF_HUB_DISABLE_XET", "HTTPS_PROXY", "HTTP_PROXY")\n                   if k in os.environ}\n            log.info("Hub client: %s env=%s", versions, env)\n        except Exception:\n            pass\n        self.inspect_token()\n        info = self._call("repository inspection", self.api.repo_info,\n                          repo_id=self.repo_id, repo_type="model", missing_ok=True)\n        if info is None:\n            self._call("repository setup", self.api.create_repo, repo_id=self.repo_id,\n                       repo_type="model", private=True, exist_ok=True)\n            info = self._call("repository inspection", self.api.repo_info,\n                              repo_id=self.repo_id, repo_type="model")\n        if not info.private:\n            raise RuntimeError("Checkpoint repository must be private; choose a private repo or change its visibility on Hugging Face")\n        super().preflight()\n\n    def put(self, local, relative):\n        self._call("upload", self.api.upload_file, repo_id=self.repo_id, repo_type="model",\n                   path_or_fileobj=str(local), path_in_repo=self._path(relative),\n                   commit_message=f"Save {self.prefix}/{relative}", run_as_future=False)\n        if self._files is not None:\n            self._files.add(relative)\n\n    def put_many(self, items, lane="files"):\n        # One commit for many files: Hub limits commits separately from API requests.\n        items = list(items)\n        if not items:\n            return\n        operations = [self._commit_add(path_in_repo=self._path(relative), path_or_fileobj=str(local))\n                      for local, relative in items]\n        self._bounded(lane, "batch upload", lambda: self._call(\n            "batch upload", self.api.create_commit, repo_id=self.repo_id, repo_type="model",\n            operations=operations, commit_message=f"Save {len(items)} files under {self.prefix}",\n            run_as_future=False))\n        if self._files is not None:\n            self._files.update(relative for _, relative in items)\n\n    def get(self, relative, local):\n        import shutil\n        local = Path(local)\n        local.parent.mkdir(parents=True, exist_ok=True)\n        # local_dir avoids retaining another whole archive in the global Hub cache.\n        # Download sidecar metadata and any credentials remain outside run artifacts.\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix="hub-download-") as d:\n            downloaded = self._call("download", self._download, repo_id=self.repo_id,\n                                    repo_type="model", filename=self._path(relative),\n                                    token=self._token, local_dir=d)\n            Path(downloaded).replace(local)\n\n    def names(self):\n        if self._files is None:\n            files = self._call("file listing", self.api.list_repo_files,\n                               repo_id=self.repo_id, repo_type="model")\n            prefix = self.prefix + "/"\n            self._files = {p[len(prefix):] for p in files if p.startswith(prefix)}\n        return sorted(self._files)\n\n    # Hub commits can publish files and the pointer atomically without a tar copy.\n    archive_snapshots = False\n\n    def backup(self, run, checkpoint):\n        import shutil\n        run, checkpoint = Path(run), Path(checkpoint)\n        if not verify_checkpoint(checkpoint):\n            raise RuntimeError("Refusing to upload incomplete checkpoint")\n        with tempfile.TemporaryDirectory(dir=run.parent) as d:\n            # Model/optimizer files are immutable while training waits here.\n            # Log handlers and TensorBoard may still write in background threads.\n            paths, files = {}, {}\n            for path in sorted(run.rglob("*")):\n                relative = path.relative_to(run)\n                if not path.is_file() or relative.parts[0] == ".lock" or path.name.endswith(".tmp"):\n                    continue\n                if relative.parts[0] == "init_adapter":\n                    # Parent weights are immutable in their source repo; lineage is in init_source.json.\n                    continue\n                if relative.parts[0].startswith("checkpoint-") and relative.parts[0] != checkpoint.name:\n                    continue\n                source = path\n                if relative.parts[0] in ("tensorboard", "monitoring") or str(relative) in ("run.log", "stacks.log"):\n                    source = contained(Path(d) / "evidence", str(relative))\n                    source.parent.mkdir(parents=True, exist_ok=True)\n                    shutil.copy2(path, source)\n                paths[str(relative)] = source\n                files[str(relative)] = sha256(source)\n            snapshot = f"snapshots/{checkpoint.name}-{fingerprint(files)[:20]}"\n            info = dict(format="files-v1", checkpoint=checkpoint.name,\n                        step=int(checkpoint.name.split("-")[-1]),\n                        files={name: dict(path=f"{snapshot}/{name}", sha256=digest)\n                               for name, digest in files.items()})\n            pointer = Path(d) / "latest.json"\n            atomic_json(pointer, info)\n            operations = [self._commit_add(path_in_repo=self._path(entry["path"]),\n                          path_or_fileobj=str(paths[name])) for name, entry in info["files"].items()]\n            operations.append(self._commit_add(path_in_repo=self._path("latest.json"),\n                                               path_or_fileobj=str(pointer)))\n            # If this times out, the commit may still land later (atomically, pointer included);\n            # the lane guard stops a newer checkpoint racing it until it finishes.\n            self._bounded("checkpoint", "checkpoint commit", lambda: self._call(\n                "checkpoint commit", self.api.create_commit, repo_id=self.repo_id,\n                repo_type="model", operations=operations,\n                commit_message=f"Checkpoint {self.prefix}: {checkpoint.name}", run_as_future=False))\n        if self._files is not None:\n            self._files.update(entry["path"] for entry in info["files"].values())\n            self._files.add("latest.json")\n\n    def restore(self, run):\n        import shutil\n        run = Path(run)\n        if "latest.json" not in self.names():\n            return super().restore(run)\n        with tempfile.TemporaryDirectory(dir=run.parent) as d:\n            pointer = Path(d) / "latest.json"\n            self.get("latest.json", pointer)\n            info = read_json(pointer)\n            if info.get("format") != "files-v1":\n                return super().restore(run)  # Existing tar snapshots remain supported.\n            local = latest_checkpoint(run)\n            if local and int(local.name.split("-")[-1]) >= info["step"]:\n                return\n            staging = Path(d) / "staging"\n            staging.mkdir()\n            for name, entry in info["files"].items():\n                target = contained(staging, name)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                self.get(entry["path"], target)\n                if sha256(target) != entry["sha256"]:\n                    raise RuntimeError(f"Remote snapshot checksum mismatch: {name}")\n            checkpoint = contained(staging, info["checkpoint"])\n            if not verify_checkpoint(checkpoint):\n                raise RuntimeError("Remote checkpoint missing completion marker")\n            # Rename downloaded files on the same filesystem; no second full copy.\n            for path in sorted(staging.rglob("*"), key=lambda p: (p.name == "COMPLETE.json", str(p))):\n                if path.is_file():\n                    destination = contained(run, str(path.relative_to(staging)))\n                    destination.parent.mkdir(parents=True, exist_ok=True)\n                    path.replace(destination)\n\n    def publish_snapshot(self, archive, remote_name, pointer):\n        # The archive and pointer become visible in one atomic Hub commit.\n        # create_commit returns only after uploading blobs and committing succeeds.\n        operations = [self._commit_add(path_in_repo=self._path(remote_name), path_or_fileobj=str(archive)),\n                      self._commit_add(path_in_repo=self._path("latest.json"), path_or_fileobj=str(pointer))]\n        self._call("checkpoint commit", self.api.create_commit, repo_id=self.repo_id,\n                   repo_type="model", operations=operations,\n                   commit_message=f"Checkpoint {self.prefix}: {remote_name}", run_as_future=False)\n        if self._files is not None:\n            self._files.update((remote_name, "latest.json"))\n\n\ndef plot_logs(run):\n    """Keep logs canonical; plots are derived and can be recreated after interruption."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    run = Path(run)\n    path = run / "metrics.jsonl"\n    if not path.exists():\n        return\n    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]\n    import csv\n    fields = sorted(set().union(*(r.keys() for r in records)))\n    with (run / "metrics.csv").open("w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fields)\n        w.writeheader()\n        w.writerows(records)\n    fig, axes = plt.subplots(2, 1, figsize=(10, 7))\n    for key in ("loss", "eval_loss", "full_validation_loss"):\n        # A resumed run may repeat steps; retain latest measurement per step/key.\n        points = {r["step"]: r[key] for r in records if isinstance(r.get(key), (int, float))}\n        if points:\n            xy = sorted(points.items())\n            axes[0].plot([x for x, _ in xy], [y for _, y in xy], label=key)\n    axes[0].set(xlabel="Optimizer step", ylabel="Token cross entropy")\n    if axes[0].lines:\n        axes[0].legend()\n    points = {r["step"]: r["learning_rate"] for r in records if isinstance(r.get("learning_rate"), (int, float))}\n    xy = sorted(points.items())\n    axes[1].plot([x for x, _ in xy], [y for _, y in xy])\n    axes[1].set(xlabel="Optimizer step", ylabel="Learning rate")\n    fig.tight_layout()\n    fig.savefig(run / "loss_and_lr.png", dpi=160)\n    plt.close(fig)\n')
(ROOT/'screening_recovery.py').write_text('"""Back up preparation separately from training, without changing trainer code.\n\nThe notebook embeds this supervisor while retaining the smoke-tested trainer commit.\nSnapshots contain the preparation recipe, quality records and live shortlist WAVs.\n"""\nimport argparse\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport time\n\nLOG = logging.getLogger(\'screening_recovery\')\n\n\ndef compatible(root, recipe):\n    for name in (\'dataset_source.json\', \'selection_progress.json\', \'recovery_recipe.json\'):\n        path = root/name\n        if path.is_file():\n            data = json.loads(path.read_text())\n            return data.get(\'recipe\', data) == recipe\n    return False\n\n\ndef validate_pool(root):\n    from orpheus_utils import contained, sha256\n    if (root/\'dataset_source.json\').is_file():\n        return  # Final selection supersedes the disposable screening shortlist.\n    state = root/\'tts_scan_state.json\'\n    if state.exists():\n        for row in json.loads(state.read_text()).get(\'pool\', []):\n            path = contained(root, row[\'screening_path\'])\n            if not path.is_file() or sha256(path) != row[\'sha256\']:\n                raise RuntimeError(\'Shortlist changed during backup; retry next interval\')\n\n\ndef copy_cache(source, destination, recipe):\n    """Copy only owned preparation files; never environments, secrets or raw shards."""\n    from orpheus_utils import atomic_json\n    destination.mkdir(parents=True, exist_ok=True)\n    # Read cursor first, then its referenced audio. Validation catches concurrent deletion.\n    names = [\'tts_scan_state.json\', \'selection_progress.json\', \'dataset_source.json\',\n             \'train.csv\', \'validation.csv\', \'parts\', \'quality_cache\', \'screening\', \'audio\']\n    for name in names:\n        path = source/name\n        if path.is_symlink():\n            raise ValueError(\'Symlink in preparation cache\')\n        if path.is_dir():\n            for item in path.rglob(\'*\'):\n                if item.is_symlink():\n                    raise ValueError(\'Symlink in preparation cache\')\n                if item.is_file() and \'.tmp\' not in item.name:\n                    target = destination/item.relative_to(source)\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    try:\n                        shutil.copyfile(item, target)\n                    except FileNotFoundError:\n                        # A discarded candidate is optional; live cursor references checked below.\n                        pass\n        elif path.is_file():\n            shutil.copyfile(path, destination/name)\n    atomic_json(destination/\'recovery_recipe.json\', {\'recipe\': recipe})\n    validate_pool(destination)\n\n\ndef restore_inputs(cfg, recipe, destination, inputs):\n    """Kaggle\'s previous-version output is the bridge for the already-running job."""\n    from orpheus_utils import atomic_json\n    if (destination/\'dataset_source.json\').exists() or (destination/\'recovery_recipe.json\').exists():\n        return False\n    candidates = sorted(Path(inputs).glob(\'**/orpheus-mixed/datasets/*\'))\n    for source in candidates:\n        if source.is_dir() and compatible(source, recipe):\n            with tempfile.TemporaryDirectory(dir=destination.parent) as tmp:\n                stage = Path(tmp)/\'cache\'\n                copy_cache(source, stage, recipe)\n                shutil.copytree(stage, destination, dirs_exist_ok=True)\n            evidence = dict(source=str(source), completed_shards=len(json.loads(\n                (destination/\'tts_scan_state.json\').read_text()).get(\'done\', []))\n                if (destination/\'tts_scan_state.json\').exists() else 0)\n            atomic_json(destination/\'recovery_import.json\', evidence)\n            LOG.info(\'Previous-session screening restored: %s\', evidence)\n            return True\n    return False\n\n\ndef publish(cfg, recipe, root, api=None):\n    from huggingface_hub import HfApi, CommitOperationAdd\n    from orpheus_utils import fingerprint, sha256\n    if not root.exists():\n        return\n    prefix = \'runs/\'+cfg[\'run_id\']+\'/screening_recovery/\'\n    with tempfile.TemporaryDirectory(dir=Path(cfg[\'work_dir\'])) as tmp:\n        stage = Path(tmp)/\'cache\'\n        copy_cache(root, stage, recipe)\n        archive = Path(tmp)/\'cache.tar.gz\'\n        with tarfile.open(archive, \'w:gz\', compresslevel=1) as handle:\n            handle.add(stage, arcname=\'cache\')\n        digest = sha256(archive)\n        # Immutable blob and pointer are committed together; failed uploads leave old pointer intact.\n        relative = prefix+\'archives/\'+digest+\'.tar.gz\'\n        pointer = Path(tmp)/\'latest.json\'\n        pointer.write_text(json.dumps(dict(recipe=recipe, recipe_id=fingerprint(recipe),\n            archive=relative, sha256=digest, timestamp=time.time())))\n        (api or HfApi()).create_commit(repo_id=cfg[\'hf_repo_id\'], repo_type=\'model\',\n            operations=[CommitOperationAdd(path_in_repo=relative,path_or_fileobj=str(archive)),\n                        CommitOperationAdd(path_in_repo=prefix+\'latest.json\',path_or_fileobj=str(pointer))],\n            commit_message=\'Save resumable screening progress\')\n        LOG.info(\'Screening backup committed (%d MiB)\', archive.stat().st_size//2**20)\n\n\ndef restore_remote(cfg, recipe, destination):\n    from huggingface_hub import hf_hub_download\n    from huggingface_hub.utils import EntryNotFoundError\n    from orpheus_utils import sha256\n    pointer = \'runs/\'+cfg[\'run_id\']+\'/screening_recovery/latest.json\'\n    try:\n        data = json.loads(Path(hf_hub_download(cfg[\'hf_repo_id\'],pointer)).read_text())\n    except EntryNotFoundError:\n        return False\n    if data[\'recipe\'] != recipe:\n        raise RuntimeError(\'Remote screening recipe differs; refusing cache reuse\')\n    with tempfile.TemporaryDirectory(dir=destination.parent) as tmp:\n        archive = Path(hf_hub_download(cfg[\'hf_repo_id\'],data[\'archive\'],local_dir=tmp))\n        if sha256(archive) != data[\'sha256\']:\n            raise RuntimeError(\'Screening archive checksum mismatch\')\n        unpack = Path(tmp)/\'unpack\'\n        with tarfile.open(archive) as handle:\n            for member in handle.getmembers():\n                path = Path(member.name)\n                if path.is_absolute() or \'..\' in path.parts or not path.parts or path.parts[0] != \'cache\' or not (member.isfile() or member.isdir()):\n                    raise ValueError(\'Unsafe screening archive member\')\n            handle.extractall(unpack, filter=\'data\')\n        validate_pool(unpack/\'cache\')\n        # Copy complete snapshot, preserving no stale cursor from a different snapshot.\n        if destination.exists():\n            shutil.rmtree(destination)\n        shutil.copytree(unpack/\'cache\',destination)\n    LOG.info(\'Restored screening progress from Hugging Face\')\n    return True\n\n\ndef supervise(cfg, command, inputs=\'/kaggle/input\', interval=600, hours=None, require_cache=False):\n    from mixed_data import recipe_for\n    from orpheus_utils import fingerprint\n    recipe = recipe_for(cfg)\n    parent = Path(cfg[\'work_dir\'])/\'datasets\'\n    parent.mkdir(parents=True,exist_ok=True)\n    root = parent/fingerprint(recipe)\n    if not root.exists():\n        if not restore_remote(cfg,recipe,root):\n            imported = restore_inputs(cfg,recipe,root,inputs)\n            if require_cache and not imported:\n                raise RuntimeError("Previous screening outputs unavailable; refusing to start from scratch. Wait for the source notebook to finish and save outputs.")\n    # Commit imported progress BEFORE spending any GPU time screening again.\n    if root.exists():\n        publish(cfg,recipe,root)\n    started = time.monotonic()\n    process = subprocess.Popen(command,start_new_session=True)\n    last = started\n    stopped = False\n    try:\n        while process.poll() is None:\n            if time.monotonic()-last >= interval:\n                try:\n                    publish(cfg,recipe,root)\n                except Exception as exc:\n                    LOG.error(\'Screening backup failed (%s); local progress retained, retry in next interval\',type(exc).__name__)\n                last = time.monotonic()\n            if hours and time.monotonic()-started >= hours*3600:\n                LOG.warning(\'Session deadline: stopping process group, then saving screening progress\')\n                os.killpg(process.pid,signal.SIGINT)\n                try:\n                    process.wait(timeout=120)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid,signal.SIGTERM)\n                    try:\n                        process.wait(timeout=30)\n                    except subprocess.TimeoutExpired:\n                        os.killpg(process.pid,signal.SIGKILL);process.wait()\n                stopped = True\n                break\n            time.sleep(5)\n    finally:\n        if process.poll() is None:\n            os.killpg(process.pid,signal.SIGTERM)\n            process.wait(timeout=30)\n        # Failure is deliberately visible: never claim a successful durable save without one.\n        publish(cfg,recipe,root)\n    return 0 if stopped else process.returncode\n\n\nif __name__ == \'__main__\':\n    logging.basicConfig(level=logging.INFO,format=\'%(asctime)s %(levelname)s %(message)s\')\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--config\',required=True)\n    parser.add_argument(\'--hours\',type=float)\n    parser.add_argument(\'--inputs\',default=\'/kaggle/input\')\n    parser.add_argument(\'--require-cache\',action=\'store_true\')\n    parser.add_argument(\'command\',nargs=argparse.REMAINDER)\n    args=parser.parse_args()\n    command=args.command[1:] if args.command[:1]==[\'--\'] else args.command\n    if not command: parser.error(\'Missing child command\')\n    sys.exit(supervise(json.loads(Path(args.config).read_text()),command,args.inputs,hours=args.hours,require_cache=args.require_cache))\n')
sys.path.insert(0,str(ROOT))
cfg=json.loads('{"run_id": "mixed-r32-lr2e5-seed3407-stage2-primary", "model_id": "unsloth/orpheus-3b-0.1-pretrained", "model_revision": "4ff34a809af9d39dc4e023edb2d3bb2a60919411", "codec_revision": "d73ad176a12188fcf4f360ba3bf2c2fbbe8f58ec", "download_url": null, "work_dir": "/kaggle/working/orpheus-mixed", "checkpoint_backend": "huggingface", "hf_repo_id": "hassan-31x/orpheus-ft-aslp-urdutts", "data_dir": null, "train_manifest": null, "validation_manifest": null, "test_manifest": null, "speaker_column": null, "precision": "4bit", "max_length": 2048, "epochs": 1, "max_steps": -1, "micro_batch": 1, "grad_accum": 8, "learning_rate": 2e-05, "rank": 32, "lora_alpha": 64, "rslora": false, "weight_decay": 0.001, "warmup_ratio": 0.03, "scheduler": "cosine", "sampling": "random", "objective": "all", "dedup": "none", "seed": 3407, "chunk_size": 100, "save_steps": 100, "eval_steps": 50, "logging_steps": 10, "save_total_limit": 3, "eval_samples": 256, "sample_count": 3, "sample_max_new_tokens": 1400, "sample_temperature": 0.6, "sample_top_p": 0.95, "sample_repetition_penalty": 1.1, "sample_at_start": true, "sample_max_seconds": 240, "session_hours": 10.5, "minimum_train_hours": 0, "maximum_clip_seconds": 18, "invalid_row_policy": "skip", "maximum_rejected_train_fraction": 0.02, "enforce_retention_limits": true, "optimizer": "adamw_torch", "test_train_clips": null, "test_validation_clips": null, "init_from_run": null, "dataset_id": "kaab4321/UrduTTS", "dataset_revision": "a012374328f140d2647e9138e6ceae85d4ee9f2c", "dataset_text_column": "transcription", "dataset_include_test": false, "init_source": {"repo_id": "hassan-31x/orpheus-urdutts-checkpoints", "revision": "8494444672fd2e1b3e44758a24b17ef70b8e5ad4", "run_id": "urdutts-r32-lr1e4-seed3407-epoch1", "checkpoint": "checkpoint-5029"}, "mixture": {"aslp_id": "ASLP-lab/UrduSpeech", "aslp_revision": "16dd380cfd9049a3db7f06a98e878086916bf833", "aslp_fraction": 0.6, "max_aslp_hours": 3.0, "minimum_aslp_hours": 0.1, "minimum_core_fraction": 0.65, "category_caps": {"drama": 0.12, "news": 0.03, "roadside_interview": 0.2}, "candidate_limits": {"interview": 2000, "podcast": 2000, "drama": 2000, "news": 500, "roadside_interview": 1200}, "speaker_max_seconds": 180, "validation_fraction": 0.1, "minimum_confidence": 0.98, "minimum_seconds": 2, "maximum_seconds": 18, "maximum_chars": 210, "minimum_urdu_letter_fraction": 0.95, "maximum_cer": 0.15, "maximum_wer": 0.35, "minimum_dnsmos_ovrl": 3.1, "minimum_dnsmos_sig": 3.4, "minimum_dnsmos_bak": 3.5, "maximum_clipped_fraction": 0.001, "minimum_rms_db": -38, "maximum_rms_db": -10, "minimum_vad_fraction": 0.55, "tts_shortlist_multiplier": 3, "tts_scan_limit": null, "validation_clips_per_source": 100, "asr_model": "Systran/faster-whisper-large-v3", "asr_revision": "edaa852ec7e145841d8ffdb056a99866b5f0a478", "quality_device": "cuda", "curriculum_aslp_fractions": [0.5, 0.6, 0.7]}}')
recipe=json.loads('{"dataset_id": "kaab4321/UrduTTS", "revision": "a012374328f140d2647e9138e6ceae85d4ee9f2c", "aslp_id": "ASLP-lab/UrduSpeech", "aslp_revision": "16dd380cfd9049a3db7f06a98e878086916bf833", "mixture": {"aslp_id": "ASLP-lab/UrduSpeech", "aslp_revision": "16dd380cfd9049a3db7f06a98e878086916bf833", "aslp_fraction": 0.6, "max_aslp_hours": 3.0, "minimum_aslp_hours": 0.1, "minimum_core_fraction": 0.65, "category_caps": {"drama": 0.12, "news": 0.03, "roadside_interview": 0.2}, "candidate_limits": {"interview": 2000, "podcast": 2000, "drama": 2000, "news": 500, "roadside_interview": 1200}, "speaker_max_seconds": 180, "validation_fraction": 0.1, "minimum_confidence": 0.98, "minimum_seconds": 2, "maximum_seconds": 18, "maximum_chars": 210, "minimum_urdu_letter_fraction": 0.95, "maximum_cer": 0.15, "maximum_wer": 0.35, "minimum_dnsmos_ovrl": 3.1, "minimum_dnsmos_sig": 3.4, "minimum_dnsmos_bak": 3.5, "maximum_clipped_fraction": 0.001, "minimum_rms_db": -38, "maximum_rms_db": -10, "minimum_vad_fraction": 0.55, "tts_shortlist_multiplier": 3, "tts_scan_limit": null, "validation_clips_per_source": 100, "asr_model": "Systran/faster-whisper-large-v3", "asr_revision": "edaa852ec7e145841d8ffdb056a99866b5f0a478", "quality_device": "cuda", "curriculum_aslp_fractions": [0.5, 0.6, 0.7]}, "seed": 3407, "test_train_clips": null, "test_validation_clips": null, "dns_revision": "591184a9fcb2cbdec02520fed81a32bbbf9d73ff", "source_sha256": {"mixed_data.py": "b0d8fe9a793fa89c5f5a154d8b1140634858b4bd59a6980dea45362a0845884e", "quality.py": "0ec7f0079f8db82c0bc20a14619266e1899854b186fcbf2b408eaeb567e23457", "mixture.py": "a5caf258a80b2acb99a77c377e59a8f88d707413909a399476e87870f33a0646", "urdutts_data.py": "38cdecf7b519f0c6ecb82bf7ea66e73ab5d707cc29db4f40c112ada47f42ab4f"}, "packages": {"numpy": "2.5.4", "scipy": "1.18.1", "soundfile": "0.14.0", "pyarrow": "26.0.1", "onnxruntime": "1.31.1", "faster-whisper": "1.2.1", "ctranslate2": "4.8.5"}}')
from screening_recovery import restore_inputs,publish,restore_remote
from orpheus_utils import fingerprint,atomic_json
logging.basicConfig(level=logging.INFO)
work=Path(cfg['work_dir']);work.mkdir(parents=True,exist_ok=True)
destination=work/'datasets'/fingerprint(recipe);destination.parent.mkdir(parents=True,exist_ok=True)
if not restore_inputs(cfg,recipe,destination,'/kaggle/input'):
    raise RuntimeError('No matching saved screening output found. Do not start another GPU run.')
publish(cfg,recipe,destination)
verify=work/'verified-download'
restore_remote(cfg,recipe,verify)
state=json.loads((verify/'tts_scan_state.json').read_text()) if (verify/'tts_scan_state.json').exists() else {}
receipt=dict(status='recovery_verified',completed_shards=state.get('done',[]),quality_records=len(list((verify/'quality_cache').glob('*.json'))))
atomic_json(Path('/kaggle/working/recovery_verified.json'),receipt)
print('RECOVERY VERIFIED:',len(receipt['completed_shards']),'completed shards;',receipt['quality_records'],'cached quality results; uploaded and downloaded successfully')
